# Notebook 5W — Wikimedia Context Bridge for Women Discourse

This notebook prepares reusable Wikimedia context artifacts for the women-discourse context-augmented modelling stage.

It creates:

1. `wiki_articles.csv` — raw Wikimedia article text for gender-related subgroup article banks.
2. `wiki_summaries.csv` — one Llama-generated summary per article.
3. `subgroup_summary_bank.json` — summaries grouped by gender subgroup.
4. `context_mapped_examples.parquet` — one row per comment-subgroup example with retrieved context.
5. `context_token_length_report.csv` — RoBERTa token-length diagnostics.

The mapping is:

```text
(comment, gender subgroup)
    ↓
retrieve top-k summaries from that subgroup's Wikimedia article pool
    ↓
context input text
```

This women version intentionally keeps only:

```python
["women", "men", "non_binary"]
```

so `prefer_not_to_say` and `self_describe` are not included in the context dataset.


## 1. Imports and Configuration

In [21]:
import ast
import json
import time
import re
from pathlib import Path
from typing import Dict, List, Any

import numpy as np
import pandas as pd
import requests

from tqdm.auto import tqdm
from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 300)

DATA_DIR = Path("/home/shayan/Distributional-Hate-Speech-Prediction/notebooks/data")
OUTPUT_DIR = Path("women_context_artifacts")
OUTPUT_DIR.mkdir(exist_ok=True)

WOMEN_PROCESSED_PATH = DATA_DIR / "women_processed (1).parquet"

DISCOURSE = "women"
ALLOWED_SUBGROUPS = ["women", "men", "non_binary"]
SUBGROUP_HEADER = "ANNOTATOR GENDER"

MODEL_NAME_FOR_TOKEN_LENGTH = "roberta-base"

TOP_K_SUMMARIES = 1
SUMMARY_MIN_WORDS = 50
SUMMARY_MAX_WORDS = 90

SBERT_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# Ollama local Llama settings. Change OLLAMA_MODEL if your local model name differs.
OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "llama3.1:8b"

print("Output directory:", OUTPUT_DIR.resolve())
print("Allowed subgroups:", ALLOWED_SUBGROUPS)


Output directory: /home/shayan/Distributional-Hate-Speech-Prediction/notebooks/women_models/women_context_artifacts
Allowed subgroups: ['women', 'men', 'non_binary']


## 2. Define Gender-Subgroup-to-Wikimedia Article Bank


In [3]:
ARTICLE_BANK = {
    "men": [
        # Primary articles
        "Masculinity",
        "Hegemonic masculinity",
        "Male privilege",
        # Secondary articles
        "Gender role",
        "Toxic masculinity",
    ],
    "women": [
        # Primary articles
        "Misogyny",
        "Sexism",
        "Patriarchy",
        # Secondary articles
        "Violence against women",
        "Gender inequality",
    ],
    "non_binary": [
        # Primary articles
        "Non-binary gender",
        "Genderqueer",
        "Gender identity",
        # Secondary articles
        "Gender diversity",
        "Third gender",
    ],
}

# Safety check: only the explicitly selected subgroups should be used downstream.
assert set(ARTICLE_BANK.keys()) == set(ALLOWED_SUBGROUPS)

all_article_titles = sorted({title for titles in ARTICLE_BANK.values() for title in titles})

print("Number of unique articles:", len(all_article_titles))
display(pd.DataFrame({"article_title": all_article_titles}))

article_bank_path = OUTPUT_DIR / "article_bank.json"
with open(article_bank_path, "w", encoding="utf-8") as f:
    json.dump(ARTICLE_BANK, f, indent=2, ensure_ascii=False)

print("Saved:", article_bank_path)


Number of unique articles: 15


,article_title
0,Gender diversity
1,Gender identity
2,Gender inequality
3,Gender role
4,Genderqueer
5,Hegemonic masculinity
6,Male privilege
7,Masculinity
8,Misogyny
9,Non-binary gender


Saved: women_context_artifacts/article_bank.json


## 3. Fetch Wikimedia Article Text

In [5]:
def fetch_wikipedia_full_extract(title: str, language: str = "en") -> Dict[str, Any]:
    url = f"https://{language}.wikipedia.org/w/api.php"

    params = {
        "action": "query",
        "prop": "extracts|info",
        "explaintext": "1",
        "exsectionformat": "plain",
        "inprop": "url",
        "redirects": "1",
        "titles": title,
        "format": "json",
    }

    response = requests.get(
        url,
        params=params,
        headers={"User-Agent": "distributional-hate-speech-dissertation/1.0"},
        timeout=30,
    )

    if response.status_code != 200:
        return {
            "requested_title": title,
            "resolved_title": None,
            "page_url": None,
            "raw_text": None,
            "status_code": response.status_code,
            "error": response.text[:500],
        }

    data = response.json()
    pages = data.get("query", {}).get("pages", {})
    page = next(iter(pages.values()))

    return {
        "requested_title": title,
        "resolved_title": page.get("title"),
        "page_url": page.get("fullurl"),
        "raw_text": page.get("extract"),
        "status_code": response.status_code,
        "error": None,
    }


def clean_article_text(text: str) -> str:
    if text is None:
        return None

    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = text.strip()

    stop_markers = [
        "\nReferences\n",
        "\nSee also\n",
        "\nExternal links\n",
        "\nFurther reading\n",
    ]

    for marker in stop_markers:
        if marker in text:
            text = text.split(marker)[0].strip()

    return text


In [6]:
articles_path = OUTPUT_DIR / "wiki_articles.csv"

if articles_path.exists():
    print("Loading existing article file:", articles_path)
    wiki_articles_df = pd.read_csv(articles_path)
else:
    article_rows = []

    for title in tqdm(all_article_titles):
        result = fetch_wikipedia_full_extract(title)
        result["raw_text"] = clean_article_text(result["raw_text"])
        result["char_count"] = len(result["raw_text"]) if isinstance(result["raw_text"], str) else 0
        article_rows.append(result)
        time.sleep(0.2)

    wiki_articles_df = pd.DataFrame(article_rows)
    wiki_articles_df.to_csv(articles_path, index=False)

print("Articles:", wiki_articles_df.shape)
display(wiki_articles_df[["requested_title", "resolved_title", "char_count", "page_url", "error"]])
print("Saved:", articles_path)


Loading existing article file: women_context_artifacts/wiki_articles.csv
Articles: (15, 7)


,requested_title,resolved_title,char_count,page_url,error
0,Gender diversity,Gender diversity,6187,https://en.wikipedia.org/wiki/Gender_diversity,NaN
1,Gender identity,Gender identity,37824,https://en.wikipedia.org/wiki/Gender_identity,NaN
2,Gender inequality,Gender inequality,83581,https://en.wikipedia.org/wiki/Gender_inequality,NaN
3,Gender role,Gender role,88089,https://en.wikipedia.org/wiki/Gender_role,NaN
4,Genderqueer,Non-binary,27239,https://en.wikipedia.org/wiki/Non-binary,NaN
5,Hegemonic masculinity,Hegemonic masculinity,40850,https://en.wikipedia.org/wiki/Hegemonic_masculinity,NaN
6,Male privilege,Male privilege,14891,https://en.wikipedia.org/wiki/Male_privilege,NaN
7,Masculinity,Masculinity,48488,https://en.wikipedia.org/wiki/Masculinity,NaN
8,Misogyny,Misogyny,37143,https://en.wikipedia.org/wiki/Misogyny,NaN
9,Non-binary gender,Non-binary,27239,https://en.wikipedia.org/wiki/Non-binary,NaN


Saved: women_context_artifacts/wiki_articles.csv


In [7]:
articles_path = OUTPUT_DIR / "wiki_articles.csv"

wiki_articles_df = pd.read_csv(articles_path)

missing_titles = wiki_articles_df.loc[
    wiki_articles_df["raw_text"].isna()
    | (wiki_articles_df["char_count"] < 500),
    "requested_title"
].tolist()

print("Missing titles:", missing_titles)

Missing titles: []


In [8]:
fixed_rows = []

for title in tqdm(missing_titles):
    result = fetch_wikipedia_full_extract(title)
    result["raw_text"] = clean_article_text(result["raw_text"])
    result["char_count"] = len(result["raw_text"]) if isinstance(result["raw_text"], str) else 0
    fixed_rows.append(result)

    time.sleep(3.0)   # slower to avoid rate limits

fixed_df = pd.DataFrame(fixed_rows)

display(fixed_df[["requested_title", "resolved_title", "char_count", "error"]])

0it [00:00, ?it/s]

KeyError: "None of [Index(['requested_title', 'resolved_title', 'char_count', 'error'], dtype='object')] are in the [columns]"

In [9]:
for _, fixed in fixed_df.iterrows():
    mask = wiki_articles_df["requested_title"] == fixed["requested_title"]

    for col in fixed_df.columns:
        wiki_articles_df.loc[mask, col] = fixed[col]

wiki_articles_df.to_csv(articles_path, index=False)

display(wiki_articles_df[["requested_title", "resolved_title", "char_count", "error"]])

,requested_title,resolved_title,char_count,error
0,Gender diversity,Gender diversity,6187,NaN
1,Gender identity,Gender identity,37824,NaN
2,Gender inequality,Gender inequality,83581,NaN
3,Gender role,Gender role,88089,NaN
4,Genderqueer,Non-binary,27239,NaN
5,Hegemonic masculinity,Hegemonic masculinity,40850,NaN
6,Male privilege,Male privilege,14891,NaN
7,Masculinity,Masculinity,48488,NaN
8,Misogyny,Misogyny,37143,NaN
9,Non-binary gender,Non-binary,27239,NaN


## 4. Inspect and Validate Fetched Articles

In [10]:
missing_articles = wiki_articles_df[
    wiki_articles_df["raw_text"].isna()
    | (wiki_articles_df["char_count"] < 500)
]

if len(missing_articles) > 0:
    print("Potentially problematic articles:")
    display(missing_articles[["requested_title", "resolved_title", "char_count", "error"]])
else:
    print("All articles look usable.")

display(
    wiki_articles_df[["requested_title", "resolved_title", "char_count"]]
    .sort_values("char_count")
)



All articles look usable.


,requested_title,resolved_title,char_count
0,Gender diversity,Gender diversity,6187
13,Toxic masculinity,Toxic masculinity,13012
6,Male privilege,Male privilege,14891
4,Genderqueer,Non-binary,27239
9,Non-binary gender,Non-binary,27239
10,Patriarchy,Patriarchy,35681
8,Misogyny,Misogyny,37143
1,Gender identity,Gender identity,37824
5,Hegemonic masculinity,Hegemonic masculinity,40850
12,Third gender,Third gender,43702


## 5. Llama Summarisation Prompt

In [11]:
SUMMARY_PROMPT_TEMPLATE = """Task:
Write a concise background-knowledge summary grounded only in the provided Wikimedia article text.

Purpose:
The summary will be used as context for a classifier analysing online discourse targeting women. It should explain relevant social context, gender norms, stereotypes, hostile framings, identity-based prejudice, or discourse patterns that may help interpret how the topic is discussed online.

Rules:
- Do not include any information not supported by the provided article text.
- Do not start the response with "this is a summary" or similar phrases.
- Use only information supported by the provided article text.
- Do not classify any specific post.
- Do not say "this is hate speech".
- Do not instruct the classifier what label to predict.
- Do not include moral judgement.
- Do not add unsupported claims.
- Avoid generic definitions unless they help explain gender-related discourse.
- Keep the summary between {min_words} and {max_words} words.
- Write one paragraph only.

Article title:
{article_title}

Article text:
{article_text}

Summary:
"""


def truncate_article_for_prompt(text: str, max_chars: int = 12000) -> str:
    if not isinstance(text, str):
        return ""
    return text[:max_chars]


def build_summary_prompt(article_title: str, article_text: str) -> str:
    return SUMMARY_PROMPT_TEMPLATE.format(
        min_words=SUMMARY_MIN_WORDS,
        max_words=SUMMARY_MAX_WORDS,
        article_title=article_title,
        article_text=truncate_article_for_prompt(article_text),
    )


sample_row = wiki_articles_df.iloc[0]
print(build_summary_prompt(sample_row["resolved_title"], sample_row["raw_text"])[:2000])


Task:
Write a concise background-knowledge summary grounded only in the provided Wikimedia article text.

Purpose:
The summary will be used as context for a classifier analysing online discourse targeting women. It should explain relevant social context, gender norms, stereotypes, hostile framings, identity-based prejudice, or discourse patterns that may help interpret how the topic is discussed online.

Rules:
- Do not include any information not supported by the provided article text.
- Do not start the response with "this is a summary" or similar phrases.
- Use only information supported by the provided article text.
- Do not classify any specific post.
- Do not say "this is hate speech".
- Do not instruct the classifier what label to predict.
- Do not include moral judgement.
- Do not add unsupported claims.
- Avoid generic definitions unless they help explain gender-related discourse.
- Keep the summary between 50 and 90 words.
- Write one paragraph only.

Article title:
Gender di

## 6. Generate Article Summaries with Local Llama / Ollama

In [12]:
def call_ollama(prompt: str, model: str = OLLAMA_MODEL, temperature: float = 0.1) -> str:
    """Call a local Ollama model.

    Make sure Ollama is running:
        ollama serve
        ollama pull llama3.1:8b

    If you do not use Ollama, replace this function with your local Llama call.
    """
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": temperature,
            "num_predict": 160,
        },
    }

    response = requests.post(OLLAMA_URL, json=payload, timeout=180)

    if response.status_code != 200:
        raise RuntimeError(f"Ollama error {response.status_code}: {response.text[:500]}")

    data = response.json()
    return data.get("response", "").strip()


def clean_summary(summary: str) -> str:
    if summary is None:
        return None

    summary = summary.strip()
    summary = re.sub(r"^Summary:\s*", "", summary, flags=re.I)
    summary = re.sub(r"\s+", " ", summary).strip()
    summary = summary.split("\n")[0].strip()
    return summary


def word_count(text: str) -> int:
    if not isinstance(text, str):
        return 0
    return len(text.split())


In [17]:
summaries_path = OUTPUT_DIR / "wiki_summaries.csv"

if summaries_path.exists():
    print("Loading existing summaries:", summaries_path)
    wiki_summaries_df = pd.read_csv(summaries_path)
else:
    summary_rows = []

    for _, row in tqdm(wiki_articles_df.iterrows(), total=len(wiki_articles_df)):
        article_title = row["resolved_title"] if pd.notna(row["resolved_title"]) else row["requested_title"]
        article_text = row["raw_text"]
        prompt = build_summary_prompt(article_title, article_text)

        try:
            summary = call_ollama(prompt)
            summary = clean_summary(summary)
            error = None
        except Exception as e:
            summary = None
            error = str(e)

        summary_rows.append({
            "requested_title": row["requested_title"],
            "resolved_title": article_title,
            "page_url": row["page_url"],
            "summary": summary,
            "summary_word_count": word_count(summary),
            "error": error,
        })

        time.sleep(0.2)

    wiki_summaries_df = pd.DataFrame(summary_rows)
    wiki_summaries_df.to_csv(summaries_path, index=False)

print("Summaries:", wiki_summaries_df.shape)
display(wiki_summaries_df[["requested_title", "resolved_title", "summary_word_count", "summary", "error"]])
print("Saved:", summaries_path)


  0%|          | 0/15 [00:00<?, ?it/s]

Summaries: (15, 6)


,requested_title,resolved_title,summary_word_count,summary,error
0,Gender diversity,Gender diversity,80,"The concept of gender diversity refers to the equitable representation of people of different genders, including men and women, as well as non-binary individuals. In fields traditionally dominated by men, such as computing, engineering, medicine, and science, initiatives aim to promote gender di...",None
1,Gender identity,Gender identity,86,"In most societies, a gender binary exists with expectations of masculinity and femininity in all aspects of sex and gender. This binary is often internalized by children from an early age, with core gender identity forming by around 3 years old. The formation of gender identity is influenced by ...",None
2,Gender inequality,Gender inequality,96,"Gender inequality refers to disparities between men and women in access to opportunities, resources, rights, and protections. Biological differences between sexes include reproductive roles, physical strength, and longevity. Psychological differences include risk-taking tendencies, aggression, a...",None
3,Gender role,Gender role,88,"Gender roles are social norms that vary across cultures, influencing a wide range of human behavior, including profession, relationships, and personal expression. Traditionally, women have been confined to the ""private"" sphere while men occupy the ""public"" sphere. Sociologists distinguish betwee...",None
4,Genderqueer,Non-binary,85,"Non-binary individuals often identify outside the traditional male/female binary, with some experiencing elements of both or neither gender norm as relevant to them. Non-binary identities can manifest in various ways, including bigender (two distinct genders), demigender (partial connection to o...",None
5,Hegemonic masculinity,Hegemonic masculinity,92,"Hegemonic masculinity refers to the culturally idealized form of manhood that justifies men's dominant position in society and subordinates women and other marginalized groups. It is characterized by traits such as violence, aggression, stoicism, and competitiveness, which are often internalized...",None
6,Male privilege,Male privilege,100,"In patriarchal societies, males hold primary power and predominate in roles of leadership, moral authority, social privilege, and control of property, granting them economic, political, social, educational, and practical advantages over women. These privileges are often invisible to holders, lea...",None
7,Masculinity,Masculinity,76,"In Western cultures, traditional masculine traits include strength, courage, independence, leadership, dominance, and assertiveness. However, standards of masculinity vary across cultures and historical periods. The concept of masculinity is socially constructed, influenced by both cultural fact...",None
8,Misogyny,Misogyny,97,"Misogyny is a form of sexism that perpetuates women's subordinate status in patriarchal societies. It can manifest in obvious and subtle ways, including violence against women, sexual harassment, and exclusion from full citizenship. Misogyny is often linked to femmephobia, the rejection of femin...",None
9,Non-binary gender,Non-binary,91,"Non-binary individuals often experience elements of both male and female gender norms as relevant to them, or neither set of norms feels relevant. This can manifest in various ways, such as identifying with multiple genders, experiencing a fluid range of genders, or rejecting traditional binary ...",None


Saved: women_context_artifacts/wiki_summaries.csv


## 7. Validate Summary Lengths

In [18]:
summary_issues = wiki_summaries_df[
    wiki_summaries_df["summary"].isna()
    | (wiki_summaries_df["summary_word_count"] < SUMMARY_MIN_WORDS)
    | (wiki_summaries_df["summary_word_count"] > SUMMARY_MAX_WORDS)
]

if len(summary_issues) > 0:
    print("Summaries outside desired length or missing:")
    display(summary_issues[["requested_title", "summary_word_count", "summary", "error"]])
else:
    print("All summaries are within the requested word range.")

display(
    wiki_summaries_df[["requested_title", "summary_word_count"]]
    .sort_values("summary_word_count")
)


Summaries outside desired length or missing:


,requested_title,summary_word_count,summary,error
2,Gender inequality,96,"Gender inequality refers to disparities between men and women in access to opportunities, resources, rights, and protections. Biological differences between sexes include reproductive roles, physical strength, and longevity. Psychological differences include risk-taking tendencies, aggression, a...",None
5,Hegemonic masculinity,92,"Hegemonic masculinity refers to the culturally idealized form of manhood that justifies men's dominant position in society and subordinates women and other marginalized groups. It is characterized by traits such as violence, aggression, stoicism, and competitiveness, which are often internalized...",None
6,Male privilege,100,"In patriarchal societies, males hold primary power and predominate in roles of leadership, moral authority, social privilege, and control of property, granting them economic, political, social, educational, and practical advantages over women. These privileges are often invisible to holders, lea...",None
8,Misogyny,97,"Misogyny is a form of sexism that perpetuates women's subordinate status in patriarchal societies. It can manifest in obvious and subtle ways, including violence against women, sexual harassment, and exclusion from full citizenship. Misogyny is often linked to femmephobia, the rejection of femin...",None
9,Non-binary gender,91,"Non-binary individuals often experience elements of both male and female gender norms as relevant to them, or neither set of norms feels relevant. This can manifest in various ways, such as identifying with multiple genders, experiencing a fluid range of genders, or rejecting traditional binary ...",None
12,Third gender,92,"In many cultures, a third gender is recognized as an identity that falls outside of the traditional binary of male and female. This concept can be found in various forms around the world, including among Native Hawaiians (māhū), Indigenous North Americans (Two-Spirit), South Asians (hijras), and...",None


,requested_title,summary_word_count
14,Violence against women,72
13,Toxic masculinity,75
7,Masculinity,76
0,Gender diversity,80
11,Sexism,81
4,Genderqueer,85
1,Gender identity,86
3,Gender role,88
10,Patriarchy,90
9,Non-binary gender,91


## 8. Build Subgroup Summary Bank

In [19]:
summary_lookup = {
    row["requested_title"]: {
        "article_title": row["requested_title"],
        "resolved_title": row["resolved_title"],
        "page_url": row["page_url"],
        "summary": row["summary"],
    }
    for _, row in wiki_summaries_df.iterrows()
}

subgroup_summary_bank = {}

for subgroup, article_titles in ARTICLE_BANK.items():
    subgroup_summary_bank[subgroup] = []

    for title in article_titles:
        if title not in summary_lookup:
            raise ValueError(f"Missing summary for article: {title}")

        item = summary_lookup[title].copy()
        item["subgroup"] = subgroup
        subgroup_summary_bank[subgroup].append(item)

bank_path = OUTPUT_DIR / "subgroup_summary_bank.json"

with open(bank_path, "w", encoding="utf-8") as f:
    json.dump(subgroup_summary_bank, f, indent=2, ensure_ascii=False)

print("Saved:", bank_path)

for subgroup, items in subgroup_summary_bank.items():
    print("\n", subgroup)
    for item in items:
        print("-", item["article_title"], "|", str(item["summary"])[:120], "...")


Saved: women_context_artifacts/subgroup_summary_bank.json

 men
- Masculinity | In Western cultures, traditional masculine traits include strength, courage, independence, leadership, dominance, and as ...
- Hegemonic masculinity | Hegemonic masculinity refers to the culturally idealized form of manhood that justifies men's dominant position in socie ...
- Male privilege | In patriarchal societies, males hold primary power and predominate in roles of leadership, moral authority, social privi ...
- Gender role | Gender roles are social norms that vary across cultures, influencing a wide range of human behavior, including professio ...
- Toxic masculinity | Toxic masculinity refers to traditional cultural masculine norms that can be harmful to men, women, and society overall. ...

 women
- Misogyny | Misogyny is a form of sexism that perpetuates women's subordinate status in patriarchal societies. It can manifest in ob ...
- Sexism | Sexism is prejudice or discrimination based on one's se

## 9. Load Processed Women Data and Expand to Comment–Subgroup Rows


In [22]:
women_df = pd.read_parquet(WOMEN_PROCESSED_PATH)

print("Women processed:", women_df.shape)
display(women_df.head(2))


Women processed: (3953, 12)


,comment_id,text,split,num_annotations,overall_counts,overall_distribution,entropy,majority_label,expected_label,subgroup_counts,subgroup_label_counts,subgroup_distributions
0,6,First off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;),train,2,"[2.0, 0.0, 0.0]","[1.0, 0.0, 0.0]",0.0,0,0.0,"{'men': 1.0, 'non_binary': None, 'prefer_not_to_say': None, 'self_describe': None, 'women': 1.0}","{'men': [1.0, 0.0, 0.0], 'non_binary': None, 'prefer_not_to_say': None, 'self_describe': None, 'women': [1.0, 0.0, 0.0]}","{'men': [1.0, 0.0, 0.0], 'non_binary': None, 'prefer_not_to_say': None, 'self_describe': None, 'women': [1.0, 0.0, 0.0]}"
1,11,"eat my fuck, bitch",validation,2,"[0.0, 1.0, 1.0]","[0.0, 0.5, 0.5]",1.0,1,1.5,"{'men': 1.0, 'non_binary': None, 'prefer_not_to_say': None, 'self_describe': None, 'women': 1.0}","{'men': [0.0, 0.0, 1.0], 'non_binary': None, 'prefer_not_to_say': None, 'self_describe': None, 'women': [0.0, 1.0, 0.0]}","{'men': [0.0, 0.0, 1.0], 'non_binary': None, 'prefer_not_to_say': None, 'self_describe': None, 'women': [0.0, 1.0, 0.0]}"


In [23]:
NUM_LABELS = 3


def ensure_dict(value):
    if isinstance(value, dict):
        return value
    if isinstance(value, str):
        return ast.literal_eval(value)
    raise TypeError(f"Expected dict or stringified dict, got {type(value)}")


def is_valid_distribution(dist, num_labels: int = NUM_LABELS, tolerance: float = 1e-5) -> bool:
    if dist is None:
        return False
    try:
        dist = [float(x) for x in dist]
    except Exception:
        return False
    if len(dist) != num_labels:
        return False
    if any(p < -tolerance for p in dist):
        return False
    return abs(sum(dist) - 1.0) < tolerance


def expand_subgroup_examples(
    comment_df: pd.DataFrame,
    allowed_subgroups: list[str] = ALLOWED_SUBGROUPS,
) -> pd.DataFrame:
    rows = []
    skipped_none = 0
    skipped_invalid = 0
    skipped_not_allowed = 0

    for _, row in comment_df.iterrows():
        subgroup_distributions = ensure_dict(row["subgroup_distributions"])
        subgroup_counts = ensure_dict(row["subgroup_counts"])

        for subgroup, target_distribution in subgroup_distributions.items():
            # This is the key women-discourse restriction:
            # keep only women, men, and non_binary; exclude prefer_not_to_say/self_describe.
            if subgroup not in allowed_subgroups:
                skipped_not_allowed += 1
                continue

            # Also require a Wikimedia article bank for that subgroup.
            if subgroup not in ARTICLE_BANK:
                skipped_not_allowed += 1
                continue

            if target_distribution is None:
                skipped_none += 1
                continue

            if not is_valid_distribution(target_distribution):
                skipped_invalid += 1
                continue

            target_distribution = [float(x) for x in target_distribution]

            rows.append({
                "experiment": DISCOURSE,
                "comment_id": row["comment_id"],
                "split": row["split"],
                "subgroup": subgroup,
                "subgroup_count": int(subgroup_counts.get(subgroup, 0)),
                "text": row["text"],
                "target_distribution": target_distribution,
                "target_majority_label": int(np.argmax(target_distribution)),
                "target_expected_label": float(np.dot(np.arange(NUM_LABELS), target_distribution)),
            })

    examples = pd.DataFrame(rows)

    print(f"{DISCOURSE}: skipped {skipped_none:,} None distributions")
    print(f"{DISCOURSE}: skipped {skipped_invalid:,} invalid distributions")
    print(f"{DISCOURSE}: skipped {skipped_not_allowed:,} not-allowed subgroups")
    print(f"{DISCOURSE}: created {len(examples):,} subgroup examples")

    return examples


examples_df = expand_subgroup_examples(women_df)

print("Expanded examples:", examples_df.shape)
display(pd.crosstab(examples_df["split"], examples_df["subgroup"]))
display(examples_df.head())


women: skipped 3,897 None distributions
women: skipped 0 invalid distributions
women: skipped 7,906 not-allowed subgroups
women: created 7,962 subgroup examples
Expanded examples: (7962, 9)


subgroup,men,non_binary,women
split,,,
test,576,16,581
train,2723,108,2734
validation,604,16,604


,experiment,comment_id,split,subgroup,subgroup_count,text,target_distribution,target_majority_label,target_expected_label
0,women,6,train,men,1,First off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;),"[1.0, 0.0, 0.0]",0,0.0
1,women,6,train,women,1,First off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;),"[1.0, 0.0, 0.0]",0,0.0
2,women,11,validation,men,1,"eat my fuck, bitch","[0.0, 0.0, 1.0]",2,2.0
3,women,11,validation,women,1,"eat my fuck, bitch","[0.0, 1.0, 0.0]",1,1.0
4,women,26,train,men,2,I'd love to rip those panties off and shove my fat cock in her ass.,"[0.5, 0.0, 0.5]",0,1.0


## 10. Embed Comments and Summaries with SBERT

In [24]:
sbert = SentenceTransformer(SBERT_MODEL_NAME)

summary_rows = []

for subgroup, items in subgroup_summary_bank.items():
    for item in items:
        summary_rows.append({
            "subgroup": subgroup,
            "article_title": item["article_title"],
            "resolved_title": item["resolved_title"],
            "page_url": item["page_url"],
            "summary": item["summary"],
        })

summary_bank_df = pd.DataFrame(summary_rows)

summary_bank_path = OUTPUT_DIR / "summary_bank_flat.csv"
summary_bank_df.to_csv(summary_bank_path, index=False)

print("Summary bank:", summary_bank_df.shape)
display(summary_bank_df)
print("Saved:", summary_bank_path)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Summary bank: (15, 5)


,subgroup,article_title,resolved_title,page_url,summary
0,men,Masculinity,Masculinity,https://en.wikipedia.org/wiki/Masculinity,"In Western cultures, traditional masculine traits include strength, courage, independence, leadership, dominance, and assertiveness. However, standards of masculinity vary across cultures and historical periods. The concept of masculinity is socially constructed, influenced by both cultural fact..."
1,men,Hegemonic masculinity,Hegemonic masculinity,https://en.wikipedia.org/wiki/Hegemonic_masculinity,"Hegemonic masculinity refers to the culturally idealized form of manhood that justifies men's dominant position in society and subordinates women and other marginalized groups. It is characterized by traits such as violence, aggression, stoicism, and competitiveness, which are often internalized..."
2,men,Male privilege,Male privilege,https://en.wikipedia.org/wiki/Male_privilege,"In patriarchal societies, males hold primary power and predominate in roles of leadership, moral authority, social privilege, and control of property, granting them economic, political, social, educational, and practical advantages over women. These privileges are often invisible to holders, lea..."
3,men,Gender role,Gender role,https://en.wikipedia.org/wiki/Gender_role,"Gender roles are social norms that vary across cultures, influencing a wide range of human behavior, including profession, relationships, and personal expression. Traditionally, women have been confined to the ""private"" sphere while men occupy the ""public"" sphere. Sociologists distinguish betwee..."
4,men,Toxic masculinity,Toxic masculinity,https://en.wikipedia.org/wiki/Toxic_masculinity,"Toxic masculinity refers to traditional cultural masculine norms that can be harmful to men, women, and society overall. These norms emphasize dominance, self-reliance, competition, and the restriction of emotion, often resulting in violence, aggression, and psychological trauma. Socialization o..."
5,women,Misogyny,Misogyny,https://en.wikipedia.org/wiki/Misogyny,"Misogyny is a form of sexism that perpetuates women's subordinate status in patriarchal societies. It can manifest in obvious and subtle ways, including violence against women, sexual harassment, and exclusion from full citizenship. Misogyny is often linked to femmephobia, the rejection of femin..."
6,women,Sexism,Sexism,https://en.wikipedia.org/wiki/Sexism,"Sexism is prejudice or discrimination based on one's sex or gender, primarily affecting women and girls. It has been linked to gender roles and stereotypes, with some individuals holding that one sex is intrinsically superior to another. Sexism can manifest as hostile sexism, benevolent sexism, ..."
7,women,Patriarchy,Patriarchy,https://en.wikipedia.org/wiki/Patriarchy,"Patriarchy is a social system where men hold primary positions of authority, dominating society through various means. Sociologists attribute this to the process of socialization, which establishes gender roles and inequity as instruments of power. Patriarchal ideology rationalizes inequality by..."
8,women,Violence against women,Violence against women,https://en.wikipedia.org/wiki/Violence_against_women,"Violence against women is a manifestation of historically unequal power relations between men and women, with men being the primary perpetrators due to societal patriarchal norms. This violence can take many forms, including rape, domestic violence, sexual harassment, and forced prostitution, of..."
9,women,Gender inequality,Gender inequality,https://en.wikipedia.org/wiki/Gender_inequality,"Gender inequality refers to disparities between men and women in access to opportunities, resources, rights, and protections. Biological differences between sexes include reproductive roles, physical strength, and longevity. Psychological differences include risk-taking tendencies, aggression, a..."


Saved: women_context_artifacts/summary_bank_flat.csv


In [25]:
summary_embeddings_path = OUTPUT_DIR / "summary_embeddings.npy"

if summary_embeddings_path.exists():
    print("Loading existing summary embeddings:", summary_embeddings_path)
    summary_embeddings = np.load(summary_embeddings_path)
else:
    summary_embeddings = sbert.encode(
        summary_bank_df["summary"].tolist(),
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    np.save(summary_embeddings_path, summary_embeddings)

print("Summary embeddings:", summary_embeddings.shape)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Summary embeddings: (15, 384)


In [26]:
unique_comments_df = (
    examples_df[["comment_id", "text"]]
    .drop_duplicates("comment_id")
    .reset_index(drop=True)
)

comment_embeddings_path = OUTPUT_DIR / "comment_embeddings.npy"
unique_comments_path = OUTPUT_DIR / "unique_comments_for_context.csv"

if comment_embeddings_path.exists() and unique_comments_path.exists():
    print("Loading existing comment embeddings:", comment_embeddings_path)
    unique_comments_df = pd.read_csv(unique_comments_path)
    comment_embeddings = np.load(comment_embeddings_path)
else:
    comment_embeddings = sbert.encode(
        unique_comments_df["text"].tolist(),
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    unique_comments_df.to_csv(unique_comments_path, index=False)
    np.save(comment_embeddings_path, comment_embeddings)

print("Unique comments:", unique_comments_df.shape)
print("Comment embeddings:", comment_embeddings.shape)


Batches:   0%|          | 0/124 [00:00<?, ?it/s]

Unique comments: (3953, 2)
Comment embeddings: (3953, 384)


## 11. Retrieve Top-k Summaries Within Each Subgroup

In [27]:
comment_id_to_embedding_index = {
    comment_id: idx
    for idx, comment_id in enumerate(unique_comments_df["comment_id"])
}

subgroup_to_summary_indices = {
    subgroup: summary_bank_df.index[summary_bank_df["subgroup"] == subgroup].tolist()
    for subgroup in ARTICLE_BANK.keys()
}


def retrieve_top_k_summaries(comment_id, subgroup: str, top_k: int = TOP_K_SUMMARIES) -> List[Dict[str, Any]]:
    comment_idx = comment_id_to_embedding_index[comment_id]
    comment_emb = comment_embeddings[comment_idx].reshape(1, -1)

    candidate_indices = subgroup_to_summary_indices[subgroup]
    candidate_embeddings = summary_embeddings[candidate_indices]

    sims = cosine_similarity(comment_emb, candidate_embeddings)[0]
    ranked_local_indices = np.argsort(-sims)[:top_k]

    results = []

    for rank, local_idx in enumerate(ranked_local_indices, start=1):
        global_idx = candidate_indices[local_idx]
        row = summary_bank_df.iloc[global_idx]

        results.append({
            "rank": rank,
            "article_title": row["article_title"],
            "resolved_title": row["resolved_title"],
            "page_url": row["page_url"],
            "summary": row["summary"],
            "similarity": float(sims[local_idx]),
        })

    return results


sample = examples_df.iloc[0]
retrieve_top_k_summaries(sample["comment_id"], sample["subgroup"])


[{'rank': 1,
  'article_title': 'Male privilege',
  'resolved_title': 'Male privilege',
  'page_url': 'https://en.wikipedia.org/wiki/Male_privilege',
  'summary': 'In patriarchal societies, males hold primary power and predominate in roles of leadership, moral authority, social privilege, and control of property, granting them economic, political, social, educational, and practical advantages over women. These privileges are often invisible to holders, leading them to attribute their status to individual merits rather than unearned advantages. The ideal masculine norm varies by society but is often associated with being white, heterosexual, stoic, wealthy, strong, tough, competitive, and autonomous. Men who deviate from this norm may not benefit fully from privilege, while those who conform to it are more likely to receive rewards and recognition.',
  'similarity': 0.10770449787378311}]

## 12. Build Context Input Text

In [28]:
def build_context_input_text(row: pd.Series, retrieved_items: List[Dict[str, Any]]) -> str:
    subgroup = row["subgroup"]
    text = row["text"]

    summaries_text = "\n\n".join(
        [
            f"{item['article_title']}:\n{item['summary']}"
            for item in retrieved_items
        ]
    )

    return (
        f"### COMMENT TO CLASSIFY\n"
        f"{text}\n\n"
        f"### {SUBGROUP_HEADER}\n"
        f"{subgroup}\n\n"
        f"### RETRIEVED BACKGROUND\n"
        f"{summaries_text}"
    )


mapped_rows = []

for _, row in tqdm(examples_df.iterrows(), total=len(examples_df)):
    retrieved_items = retrieve_top_k_summaries(
        comment_id=row["comment_id"],
        subgroup=row["subgroup"],
        top_k=TOP_K_SUMMARIES,
    )

    context_input_text = build_context_input_text(row, retrieved_items)

    mapped_rows.append({
        **row.to_dict(),
        "retrieved_article_titles": [item["article_title"] for item in retrieved_items],
        "retrieved_page_urls": [item["page_url"] for item in retrieved_items],
        "retrieved_similarities": [item["similarity"] for item in retrieved_items],
        "retrieved_summaries": [item["summary"] for item in retrieved_items],
        "context_input_text": context_input_text,
    })

context_df = pd.DataFrame(mapped_rows)

print("Context mapped examples:", context_df.shape)
display(context_df.head())


  0%|          | 0/7962 [00:00<?, ?it/s]

Context mapped examples: (7962, 14)


,experiment,comment_id,split,subgroup,subgroup_count,text,target_distribution,target_majority_label,target_expected_label,retrieved_article_titles,retrieved_page_urls,retrieved_similarities,retrieved_summaries,context_input_text
0,women,6,train,men,1,First off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;),"[1.0, 0.0, 0.0]",0,0.0,[Male privilege],[https://en.wikipedia.org/wiki/Male_privilege],[0.10770449787378311],"[In patriarchal societies, males hold primary power and predominate in roles of leadership, moral authority, social privilege, and control of property, granting them economic, political, social, educational, and practical advantages over women. These privileges are often invisible to holders, le...","### COMMENT TO CLASSIFY\nFirst off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;)\n\n### ANNOTATOR GENDER\nmen\n\n### RETRIEVED BACKGROUND\nMale privilege:\nIn patriarchal societies, males hold primary power and..."
1,women,6,train,women,1,First off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;),"[1.0, 0.0, 0.0]",0,0.0,[Misogyny],[https://en.wikipedia.org/wiki/Misogyny],[0.07197389751672745],"[Misogyny is a form of sexism that perpetuates women's subordinate status in patriarchal societies. It can manifest in obvious and subtle ways, including violence against women, sexual harassment, and exclusion from full citizenship. Misogyny is often linked to femmephobia, the rejection of femi...",### COMMENT TO CLASSIFY\nFirst off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;)\n\n### ANNOTATOR GENDER\nwomen\n\n### RETRIEVED BACKGROUND\nMisogyny:\nMisogyny is a form of sexism that perpetuates women's subo...
2,women,11,validation,men,1,"eat my fuck, bitch","[0.0, 0.0, 1.0]",2,2.0,[Hegemonic masculinity],[https://en.wikipedia.org/wiki/Hegemonic_masculinity],[0.09979891777038574],"[Hegemonic masculinity refers to the culturally idealized form of manhood that justifies men's dominant position in society and subordinates women and other marginalized groups. It is characterized by traits such as violence, aggression, stoicism, and competitiveness, which are often internalize...","### COMMENT TO CLASSIFY\neat my fuck, bitch\n\n### ANNOTATOR GENDER\nmen\n\n### RETRIEVED BACKGROUND\nHegemonic masculinity:\nHegemonic masculinity refers to the culturally idealized form of manhood that justifies men's dominant position in society and subordinates women and other marginalized g..."
3,women,11,validation,women,1,"eat my fuck, bitch","[0.0, 1.0, 0.0]",1,1.0,[Misogyny],[https://en.wikipedia.org/wiki/Misogyny],[0.10469353199005127],"[Misogyny is a form of sexism that perpetuates women's subordinate status in patriarchal societies. It can manifest in obvious and subtle ways, including violence against women, sexual harassment, and exclusion from full citizenship. Misogyny is often linked to femmephobia, the rejection of femi...","### COMMENT TO CLASSIFY\neat my fuck, bitch\n\n### ANNOTATOR GENDER\nwomen\n\n### RETRIEVED BACKGROUND\nMisogyny:\nMisogyny is a form of sexism that perpetuates women's subordinate status in patriarchal societies. It can manifest in obvious and subtle ways, including violence against women, sexu..."
4,women,26,train,men,2,I'd love to rip those panties off and shove my fat cock in her ass.,"[0.5, 0.0, 0.5]",0,1.0,[Male privilege],[https://en.wikipedia.org/wiki/Male_privilege],[0.1185816153883934],"[In patriarchal societies, males hold primary power and predominate in roles of leadership, moral authority, social privilege, and control of property, granting them economic, political, social, educational, and practical advantages over women. These privileges are often invisible to holders, le

## 13. Token Length Diagnostics

In [29]:
roberta_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME_FOR_TOKEN_LENGTH)


def count_roberta_tokens(text: str) -> int:
    return len(
        roberta_tokenizer(
            text,
            truncation=False,
            add_special_tokens=True,
        )["input_ids"]
    )


context_df["tweet_token_length"] = context_df["text"].apply(count_roberta_tokens)
context_df["context_input_token_length"] = context_df["context_input_text"].apply(count_roberta_tokens)

length_report = {
    "n_rows": len(context_df),
    "tweet_mean": context_df["tweet_token_length"].mean(),
    "tweet_median": context_df["tweet_token_length"].median(),
    "tweet_p95": context_df["tweet_token_length"].quantile(0.95),
    "tweet_max": context_df["tweet_token_length"].max(),
    "context_mean": context_df["context_input_token_length"].mean(),
    "context_median": context_df["context_input_token_length"].median(),
    "context_p95": context_df["context_input_token_length"].quantile(0.95),
    "context_max": context_df["context_input_token_length"].max(),
    "pct_context_over_256": float((context_df["context_input_token_length"] > 256).mean()),
    "pct_context_over_384": float((context_df["context_input_token_length"] > 384).mean()),
    "pct_context_over_512": float((context_df["context_input_token_length"] > 512).mean()),
}

length_report_df = pd.DataFrame([length_report])
display(length_report_df)

display(
    context_df[[
        "comment_id",
        "subgroup",
        "tweet_token_length",
        "context_input_token_length",
        "retrieved_article_titles",
        "text",
        "context_input_text",
    ]]
    .sort_values("context_input_token_length", ascending=False)
    .head(10)
)


,n_rows,tweet_mean,tweet_median,tweet_p95,tweet_max,context_mean,context_median,context_p95,context_max,pct_context_over_256,pct_context_over_384,pct_context_over_512
0,7962,29.113665,22.0,74.95,143,170.833459,170.0,223.0,296,0.011806,0.0,0.0


,comment_id,subgroup,tweet_token_length,context_input_token_length,retrieved_article_titles,text,context_input_text
331,1803,men,136,296,[Male privilege],"Then if your such an easy, sloppy slut. Maybe I should just take your leash and drag you home and put you in my playroom as my personal cocksleeve. Strip you down to some heel and bend you over my spanking bench and attach you to it. Ram my 10"" cock up your ass, pull it out and feed it to you so...","### COMMENT TO CLASSIFY\nThen if your such an easy, sloppy slut. Maybe I should just take your leash and drag you home and put you in my playroom as my personal cocksleeve. Strip you down to some heel and bend you over my spanking bench and attach you to it. Ram my 10"" cock up your ass, pull it ..."
184,926,men,135,295,[Male privilege],"Yea, forget that a girl can poke a hole in a condom and lie about being on the pill and all that bull shit. Even if they both agree that they want an abortion the father has no say. A women can give birth to another mans child and have the man take care of it for years and the women would suffer...","### COMMENT TO CLASSIFY\nYea, forget that a girl can poke a hole in a condom and lie about being on the pill and all that bull shit. Even if they both agree that they want an abortion the father has no say. A women can give birth to another mans child and have the man take care of it for years a..."
242,1322,women,140,294,[Misogyny],"Me, at work: *drops box and bends down to pick up the box* Incels: ""look at that whore. She clearly dropped the box on purpose so she had an excuse to show off her whore ass. I bet she's thinking about anal sex right now. Bending over like a slut so some chad will come over and ram his dick into...","### COMMENT TO CLASSIFY\nMe, at work: *drops box and bends down to pick up the box* Incels: ""look at that whore. She clearly dropped the box on purpose so she had an excuse to show off her whore ass. I bet she's thinking about anal sex right now. Bending over like a slut so some chad will come o..."
185,926,women,135,294,[Gender inequality],"Yea, forget that a girl can poke a hole in a condom and lie about being on the pill and all that bull shit. Even if they both agree that they want an abortion the father has no say. A women can give birth to another mans child and have the man take care of it for years and the women would suffer...","### COMMENT TO CLASSIFY\nYea, forget that a girl can poke a hole in a condom and lie about being on the pill and all that bull shit. Even if they both agree that they want an abortion the father has no say. A women can give birth to another mans child and have the man take care of it for years a..."
1497,8603,men,134,294,[Male privilege],For escaping rapists only she didn't escape them at all not in a man's world where women exist so men can have some fun and boost their own male morale by smashing women's down we see the big cheesy grins on men's faces NEWS FLASH Twelve Israelis held in Cyprus over alleged gang rape of British ...,### COMMENT TO CLASSIFY\nFor escaping rapists only she didn't escape them at all not in a man's world where women exist so men can have some fun and boost their own male morale by smashing women's down we see the big cheesy grins on men's faces NEWS FLASH Twelve Israelis held in Cyprus over alle...
733,4175,men,132,292,[Male privilege],it s not that i don t want to have sex or that anyone doesn t want to have sex with me i just am too good for girls . females are trash and doesn t deserve rights . i d be doing them a favour if i had sex with them bc i m so hot but i won t bc i m too good for them . fucking women deserve to die...,### COMMENT TO CLASSIFY\nit s not that i don t want to have sex or that anyone doesn t want to have sex with me i just am too good for girls . females are trash and doesn t deserve rights . i d be doing them a favour if i had sex with them bc i m so hot but i won t bc i m too good for them . fuc...
452,2580,men,131,291,[Male privilege

## 14. Save Final Context Artifacts

In [30]:
context_parquet_path = OUTPUT_DIR / "women_context_mapped_examples.parquet"
context_csv_path = OUTPUT_DIR / "women_context_mapped_examples.csv"
length_report_path = OUTPUT_DIR / "women_context_token_length_report.csv"

context_df.to_parquet(context_parquet_path, index=False)
context_df.to_csv(context_csv_path, index=False)
length_report_df.to_csv(length_report_path, index=False)

print("Saved:", context_parquet_path)
print("Saved:", context_csv_path)
print("Saved:", length_report_path)

print("\nAll created files:")
for path in sorted(OUTPUT_DIR.glob("*")):
    print("-", path)


Saved: women_context_artifacts/women_context_mapped_examples.parquet
Saved: women_context_artifacts/women_context_mapped_examples.csv
Saved: women_context_artifacts/women_context_token_length_report.csv

All created files:
- women_context_artifacts/article_bank.json
- women_context_artifacts/comment_embeddings.npy
- women_context_artifacts/subgroup_summary_bank.json
- women_context_artifacts/summary_bank_flat.csv
- women_context_artifacts/summary_embeddings.npy
- women_context_artifacts/unique_comments_for_context.csv
- women_context_artifacts/wiki_articles.csv
- women_context_artifacts/wiki_summaries.csv
- women_context_artifacts/women_context_mapped_examples.csv
- women_context_artifacts/women_context_mapped_examples.parquet
- women_context_artifacts/women_context_token_length_report.csv


## 15. Manual Inspection Checklist

Before moving to the model notebook, inspect:

1. Are article summaries grounded and useful?
2. Are top-k retrieved summaries reasonable for comments?
3. Are token lengths acceptable for RoBERTa?
4. Does `context_input_text` look clean?
5. Are the same comments receiving different summaries under different gender subgroups?
6. Does `examples_df["subgroup"].value_counts()` contain only `women`, `men`, and `non_binary`?

The next notebook should train context-augmented women-discourse models using:

```text
women_context_artifacts/women_context_mapped_examples.parquet
```


In [31]:
context_df[
    [
        "subgroup",
        "retrieved_article_titles",
        "retrieved_summaries"
    ]
].head(10)

,subgroup,retrieved_article_titles,retrieved_summaries
0,men,[Male privilege],"[In patriarchal societies, males hold primary power and predominate in roles of leadership, moral authority, social privilege, and control of property, granting them economic, political, social, educational, and practical advantages over women. These privileges are often invisible to holders, le..."
1,women,[Misogyny],"[Misogyny is a form of sexism that perpetuates women's subordinate status in patriarchal societies. It can manifest in obvious and subtle ways, including violence against women, sexual harassment, and exclusion from full citizenship. Misogyny is often linked to femmephobia, the rejection of femi..."
2,men,[Hegemonic masculinity],"[Hegemonic masculinity refers to the culturally idealized form of manhood that justifies men's dominant position in society and subordinates women and other marginalized groups. It is characterized by traits such as violence, aggression, stoicism, and competitiveness, which are often internalize..."
3,women,[Misogyny],"[Misogyny is a form of sexism that perpetuates women's subordinate status in patriarchal societies. It can manifest in obvious and subtle ways, including violence against women, sexual harassment, and exclusion from full citizenship. Misogyny is often linked to femmephobia, the rejection of femi..."
4,men,[Male privilege],"[In patriarchal societies, males hold primary power and predominate in roles of leadership, moral authority, social privilege, and control of property, granting them economic, political, social, educational, and practical advantages over women. These privileges are often invisible to holders, le..."
5,women,[Violence against women],"[Violence against women is a manifestation of historically unequal power relations between men and women, with men being the primary perpetrators due to societal patriarchal norms. This violence can take many forms, including rape, domestic violence, sexual harassment, and forced prostitution, o..."
6,men,[Gender role],"[Gender roles are social norms that vary across cultures, influencing a wide range of human behavior, including profession, relationships, and personal expression. Traditionally, women have been confined to the ""private"" sphere while men occupy the ""public"" sphere. Sociologists distinguish betwe..."
7,women,[Violence against women],"[Violence against women is a manifestation of historically unequal power relations between men and women, with men being the primary perpetrators due to societal patriarchal norms. This violence can take many forms, including rape, domestic violence, sexual harassment, and forced prostitution, o..."
8,men,[Male privilege],"[In patriarchal societies, males hold primary power and predominate in roles of leadership, moral authority, social privilege, and control of property, granting them economic, political, social, educational, and practical advantages over women. These privileges are often invisible to holders, le..."
9,women,[Violence against women],"[Violence against women is a manifestation of historically unequal power relations between men and women, with men being the primary perpetrators due to societal patriarchal norms. This violence can take many forms, including rape, domestic violence, sexual harassment, and forced prostitution, o..."


In [36]:
context_df["retrieved_article_titles"].apply(len).value_counts()

retrieved_article_titles
1    7962
Name: count, dtype: int64

In [37]:
context_df["retrieved_summaries"].apply(len).value_counts()

retrieved_summaries
1    7962
Name: count, dtype: int64